# BaseAgent Class Documentation

The `agents/base_agent.py` module defines the abstract base class that serves as the foundation for all specialized document analysis agents in the workstream2 agent framework.

## Module Overview

The `BaseAgent` class provides a comprehensive template and shared functionality for creating specialized agents that analyze specific document categories (requirements, design, technical specs, etc.). It implements the core agent lifecycle including context retrieval, prompt construction, LLM interaction, response parsing, and error handling.

### Design Philosophy

- **Abstract Base Class Pattern**: Uses Python's ABC (Abstract Base Class) to enforce implementation of key methods in subclasses
- **Template Method Pattern**: Defines the overall query processing workflow while allowing customization of specific steps
- **Separation of Concerns**: Isolates prompt engineering, LLM communication, and response parsing into distinct methods
- **Error Resilience**: Comprehensive exception handling with graceful degradation
- **Logging First**: Extensive logging at INFO, WARNING, and ERROR levels for observability

## Dependencies and Imports

### Standard Library

- **json**: Parses LLM responses expected in JSON format
- **logging**: Provides debug and operational logging throughout agent lifecycle
- **time**: Measures processing time for performance monitoring
- **abc**: Imports `ABC` and `abstractmethod` for abstract base class definition
- **typing**: Provides type hints (`List`, `Dict`, `Any`, `Optional`) for type safety

### Path Setup

In [5]:
import sys
from pathlib import Path
import os

# For Jupyter notebooks: add workstream2_agents directory to path
# Since __file__ is not available in notebooks, use current working directory
if '__file__' in globals():
    # If running as a script, use __file__
    workstream2_agents_path = Path(__file__).parent.parent
else:
    # If running in a notebook, find workstream2_agents directory
    # Search up from current directory to find the workstream2_agents folder
    current = Path(os.getcwd())
    workstream2_agents_path = None
    
    # Look for workstream2_agents directory (has config/settings.py)
    while current != current.parent:
        if (current / 'config' / 'settings.py').exists() and current.name == 'workstream2_agents':
            workstream2_agents_path = current
            break
        current = current.parent
    
    # Fallback: if in utils/, go up one level
    if workstream2_agents_path is None:
        cwd = Path(os.getcwd())
        workstream2_agents_path = cwd.parent if cwd.name == 'utils' else cwd

sys.path.insert(0, str(workstream2_agents_path))

### Internal Dependencies

- **agents.data_models**: Core data structures (AgentContext, AgentResponse, ConfidenceLevel, Source)
- **utils.ollama_client**: LLM client (ollama_client, OllamaException)
- **config.settings**: Global settings for model configuration

### Logging Configuration

In [6]:
import logging

logger = logging.getLogger(__name__)
# Logger name will be 'agents.base_agent' in the log hierarchy

print(logger)

<Logger __main__ (WARNING)>


## BaseAgent Class

```python
class BaseAgent(ABC):
    """Base class for all specialized agents"""
```

### Constructor

```python
def __init__(
    self,
    agent_id: str,
    name: str,
    category_id: int,
    llm_model: str = None,
    temperature: float = 0.1
)
```

#### Parameters

- **agent_id: str** (required): Unique identifier for the agent (e.g., `"requirements_agent"`)
- **name: str** (required): Human-readable agent name (e.g., `"Requirements Analysis Agent"`)
- **category_id: int** (required): Database identifier for the document category
- **llm_model: str** (optional): LLM model name, defaults to `settings.OLLAMA_MODEL`
- **temperature: float** (optional): Controls LLM randomness (0.0-1.0), default: 0.1

#### Instance Attributes

- `self.agent_id`: Stored agent identifier
- `self.name`: Stored agent name
- `self.category_id`: Stored category identifier
- `self.llm_model`: Model name
- `self.temperature`: Temperature setting
- `self.llm_client`: Reference to global ollama_client

#### Example Usage

In [7]:
from abc import ABC, abstractmethod
from typing import List, Dict, Any, Optional

# Mock imports for demonstration
class AgentContext:
    def __init__(self, project_id):
        self.project_id = project_id
        self.conversation_history = []

class ConfidenceLevel:
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"

class AgentResponse:
    def __init__(self, agent_id, agent_name, content, confidence, sources, 
                 metadata=None, processing_time_ms=0, suggested_agents=None):
        self.agent_id = agent_id
        self.agent_name = agent_name
        self.content = content
        self.confidence = confidence
        self.sources = sources
        self.metadata = metadata or {}
        self.processing_time_ms = processing_time_ms
        self.suggested_agents = suggested_agents or []

# Example subclass initialization
class RequirementsAgent:
    def __init__(self):
        self.agent_id = "requirements_agent"
        self.name = "Requirements Analysis Agent"
        self.category_id = 1
        self.temperature = 0.1
        print(f"Initialized {self.name} (ID: {self.agent_id})")

agent = RequirementsAgent()

Initialized Requirements Analysis Agent (ID: requirements_agent)


## Abstract Methods

Subclasses **must** implement these methods.

### get_system_prompt()

```python
@abstractmethod
def get_system_prompt(self) -> str:
    """
    Define agent's personality and expertise
    Must be implemented by each specialized agent
    """
    pass
```

#### Purpose

Returns the system prompt that defines the agent's role, expertise, and behavior guidelines.

#### Returns

- **str**: Multi-line system prompt text

#### Implementation Guidelines

The system prompt should:
- Define the agent's domain expertise and role
- Specify expected output format and structure
- Include guidelines for citing sources
- Set the tone and verbosity level
- Provide constraints and boundaries

#### Example Implementation

In [8]:
def get_system_prompt_example() -> str:
    return """You are a Requirements Analysis Expert specializing in software requirements documents.

Your expertise includes:
- Functional and non-functional requirements analysis
- Requirements traceability and validation
- Identifying ambiguities and conflicts in requirements

When answering queries:
1. Cite specific requirement IDs when available
2. Distinguish between mandatory (SHALL) and optional (SHOULD) requirements
3. Flag any ambiguous or incomplete requirements
4. Provide confidence levels based on source clarity

Format: Professional, precise, and structured."""

print(get_system_prompt_example())

You are a Requirements Analysis Expert specializing in software requirements documents.

Your expertise includes:
- Functional and non-functional requirements analysis
- Requirements traceability and validation
- Identifying ambiguities and conflicts in requirements

When answering queries:
1. Cite specific requirement IDs when available
2. Distinguish between mandatory (SHALL) and optional (SHOULD) requirements
3. Flag any ambiguous or incomplete requirements
4. Provide confidence levels based on source clarity

Format: Professional, precise, and structured.


### can_handle_query()

```python
@abstractmethod
def can_handle_query(self, query: str, context: AgentContext) -> float:
    """
    Determine if this agent should handle the query

    Args:
        query: User query
        context: Agent context

    Returns:
        Confidence score 0.0-1.0
    """
    pass
```

#### Purpose

Calculates a relevance score indicating how well this agent can answer the query based on its domain expertise.

#### Returns

- **float**: Confidence score between 0.0 and 1.0
  - `0.0-0.3`: Low relevance
  - `0.3-0.7`: Medium relevance
  - `0.7-1.0`: High relevance

#### Implementation Strategies

**Keyword Matching** (simple), **ML-based Classification** (advanced), **Rule-based Heuristics** (hybrid)

#### Example Implementation

In [9]:
def can_handle_query_example(query: str) -> float:
    """Example implementation using keyword matching"""
    query_lower = query.lower()

    # High confidence triggers
    high_keywords = ["requirements", "req-", "functional spec"]
    if any(phrase in query_lower for phrase in high_keywords):
        return 0.9

    # Medium confidence triggers
    medium_keywords = ["feature", "capability", "constraint"]
    if any(word in query_lower for word in medium_keywords):
        return 0.6

    # Low confidence default
    return 0.2

# Test examples
queries = [
    "What are the authentication requirements?",
    "Show me the design patterns",
    "List all features"
]

for q in queries:
    score = can_handle_query_example(q)
    print(f"Query: '{q}'")
    print(f"  Score: {score}\n")

Query: 'What are the authentication requirements?'
  Score: 0.9

Query: 'Show me the design patterns'
  Score: 0.2

Query: 'List all features'
  Score: 0.6



## Core Methods

### retrieve_context()

```python
def retrieve_context(
    self,
    query: str,
    context: AgentContext,
    top_k: int = 5
) -> List[Source]:
    """
    Retrieve relevant documents from this agent's category

    NOTE: This is a placeholder. Will be integrated with Workstream 1
    retrieval system later.
    """
```

#### Purpose

Retrieves the most relevant document chunks from the agent's specialized category to provide context for answering the query.

#### Parameters

- **query: str**: User's query text
- **context: AgentContext**: Shared agent context
- **top_k: int**: Maximum number of sources to retrieve (default: 5)

#### Returns

- **List[Source]**: Ordered list of relevant document chunks, sorted by relevance

#### Current Implementation

**Status**: Mock implementation pending Workstream 1 integration

Currently returns a single mock source with a warning logged.

#### Future Integration

Will connect to Workstream 1's vector database and retrieval pipeline for semantic search.

In [10]:
class Source:
    def __init__(self, chunk_id, document_id, filename, category_name, 
                 chunk_text, page_number=None, similarity_score=0.0):
        self.chunk_id = chunk_id
        self.document_id = document_id
        self.filename = filename
        self.category_name = category_name
        self.chunk_text = chunk_text
        self.page_number = page_number
        self.similarity_score = similarity_score

def retrieve_context_mock(query: str, category_id: int, agent_name: str) -> List[Source]:
    """Mock implementation of context retrieval"""
    print(f"⚠️  Using mock retrieval (integration pending)")

    return [
        Source(
            chunk_id=1,
            document_id=1,
            filename=f"mock_doc_{category_id}.pdf",
            category_name=agent_name,
            chunk_text=f"Mock content related to: {query}",
            page_number=1,
            similarity_score=0.85
        )
    ]

# Example usage
sources = retrieve_context_mock("authentication requirements", 1, "Requirements Analysis Agent")
for s in sources:
    print(f"Source: {s.filename} (Page {s.page_number})")
    print(f"  Similarity: {s.similarity_score}")
    print(f"  Text: {s.chunk_text}")

⚠️  Using mock retrieval (integration pending)
Source: mock_doc_1.pdf (Page 1)
  Similarity: 0.85
  Text: Mock content related to: authentication requirements


### process_query()

```python
def process_query(
    self,
    query: str,
    context: AgentContext,
    retrieved_context: Optional[List[Source]] = None
) -> AgentResponse:
    """
    Main processing logic for the agent
    """
```

#### Purpose

Orchestrates the complete query processing workflow from context retrieval through response generation. This is the main entry point for agent execution.

#### Parameters

- **query: str** (required): The user's query text
- **context: AgentContext** (required): Shared context with conversation history
- **retrieved_context: Optional[List[Source]]**: Pre-retrieved sources (if available)

#### Returns

- **AgentResponse**: Structured response including answer, confidence, sources, and metadata

#### Workflow Steps

1. **Context Retrieval**: Fetch relevant documents if not provided
2. **Prompt Construction**: Build complete prompt with system instructions, history, and context
3. **LLM Generation**: Call LLM to generate response
4. **Response Parsing**: Parse JSON output into AgentResponse
5. **Timing and Logging**: Calculate processing time and log completion

#### Error Handling

Comprehensive exception handling returns a LOW-confidence error response if any step fails, ensuring the orchestrator always receives a valid response.

In [11]:
import time
import json

def process_query_example(query: str, agent_id: str, agent_name: str) -> dict:
    """Simplified example of process_query workflow"""
    start_time = time.time()

    try:
        # Step 1: Context Retrieval
        print(f"[{agent_name}] Retrieving context...")
        sources = retrieve_context_mock(query, 1, agent_name)

        # Step 2-3: Prompt Construction & LLM Generation (simulated)
        print(f"[{agent_name}] Generating response...")
        time.sleep(0.1)  # Simulate LLM call

        # Step 4: Response Parsing (simulated)
        response_content = f"Based on the requirements, {query.lower()} is supported via OAuth 2.0."
        confidence = "high"

        # Step 5: Calculate timing
        processing_time_ms = int((time.time() - start_time) * 1000)

        print(f"[{agent_name}] Response generated ({processing_time_ms}ms)")

        return {
            "agent_id": agent_id,
            "agent_name": agent_name,
            "content": response_content,
            "confidence": confidence,
            "sources": len(sources),
            "processing_time_ms": processing_time_ms
        }

    except Exception as e:
        print(f"[{agent_name}] Error: {e}")
        return {
            "agent_id": agent_id,
            "agent_name": agent_name,
            "content": f"Error: {str(e)}",
            "confidence": "low",
            "sources": 0,
            "processing_time_ms": int((time.time() - start_time) * 1000)
        }

# Example usage
result = process_query_example(
    "What authentication methods are supported?",
    "requirements_agent",
    "Requirements Agent"
)

print("\nResponse:")
print(json.dumps(result, indent=2))

[Requirements Agent] Retrieving context...
⚠️  Using mock retrieval (integration pending)
[Requirements Agent] Generating response...
[Requirements Agent] Response generated (104ms)

Response:
{
  "agent_id": "requirements_agent",
  "agent_name": "Requirements Agent",
  "content": "Based on the requirements, what authentication methods are supported? is supported via OAuth 2.0.",
  "confidence": "high",
  "sources": 1,
  "processing_time_ms": 104
}


## Private Helper Methods

These methods support the main workflow and can be overridden by subclasses for customization.

### _build_prompt()

```python
def _build_prompt(
    self,
    query: str,
    retrieved_context: List[Source],
    context: AgentContext
) -> str:
    """Build the prompt for LLM"""
```

#### Purpose

Constructs the complete prompt sent to the LLM by assembling all necessary components.

#### Prompt Structure

1. **System Prompt**: Agent personality and expertise
2. **Conversation History**: Last 5 messages for context continuity
3. **Retrieved Context**: Formatted source documents with citations
4. **Current Query**: The user's question
5. **Output Instructions**: JSON format specification

#### JSON Output Schema

The prompt instructs the LLM to return:

```json
{
  "answer": "detailed answer with [Source N] citations",
  "confidence": "high|medium|low",
  "key_sources": ["brief description of key sources"],
  "suggests_consulting": ["agent_name1", "agent_name2"],
  "reasoning": "why you're suggesting other agents if any"
}
```

#### Example Prompt Structure

In [12]:
def build_prompt_example(query: str, sources: List[Source], system_prompt: str) -> str:
    """Example of prompt construction"""

    # Format sources
    context_text = ""
    for i, source in enumerate(sources, 1):
        context_text += f"[Source {i}] - {source.filename}"
        if source.page_number:
            context_text += f" (Page {source.page_number})"
        context_text += f"\n{source.chunk_text}\n---\n\n"

    # Build complete prompt
    prompt = f"""SYSTEM:
{system_prompt}

RELEVANT CONTEXT:
{context_text}

CURRENT QUERY:
{query}

Please provide a response in JSON format:
{{
  "answer": "your detailed answer with [Source N] citations",
  "confidence": "high|medium|low",
  "key_sources": ["source descriptions"],
  "suggests_consulting": [],
  "reasoning": ""
}}

IMPORTANT: Return ONLY valid JSON, no other text.

RESPONSE:"""

    return prompt

# Example
system_prompt = "You are a Requirements Analysis Expert."
sources = [Source(1, 1, "req.pdf", "requirements", "REQ-001: Support OAuth 2.0", 12, 0.9)]
prompt = build_prompt_example("What auth methods?", sources, system_prompt)

print(prompt[:500] + "\n... [truncated]")

SYSTEM:
You are a Requirements Analysis Expert.

RELEVANT CONTEXT:
[Source 1] - req.pdf (Page 12)
REQ-001: Support OAuth 2.0
---



CURRENT QUERY:
What auth methods?

Please provide a response in JSON format:
{
  "answer": "your detailed answer with [Source N] citations",
  "confidence": "high|medium|low",
  "key_sources": ["source descriptions"],
  "suggests_consulting": [],
  "reasoning": ""
}

IMPORTANT: Return ONLY valid JSON, no other text.

RESPONSE:
... [truncated]


### _format_context()

```python
def _format_context(self, sources: List[Source]) -> str:
    """Format retrieved sources for prompt"""
```

Converts a list of Source objects into a readable, numbered format for inclusion in the prompt.

**Format Pattern**:
```
[Source 1] - filename.pdf (Page 15)
<chunk text content>
Metadata: {"author": "John Doe"}
---
```

### _format_conversation_history()

```python
def _format_conversation_history(self, history: List[Dict]) -> str:
    """Format conversation history"""
```

Converts conversation history into readable format. **Limits to last 5 messages** to keep prompt size manageable.

**Example Output**:
```
USER: What are the authentication requirements?
ASSISTANT: The system supports OAuth 2.0 and SAML 2.0.
USER: What about token expiration?
```

In [13]:
def format_context_example(sources: List[Source]) -> str:
    """Format sources for prompt"""
    if not sources:
        return "No relevant context found."

    formatted = ""
    for i, source in enumerate(sources, 1):
        formatted += f"[Source {i}] - {source.filename}"
        if source.page_number:
            formatted += f" (Page {source.page_number})"
        else:
            formatted += " (Page N/A)"
        formatted += f"\n{source.chunk_text}\n"

        if source.metadata:
            formatted += f"Metadata: {source.metadata}\n"
        else:
            formatted += "Metadata: None\n"
        formatted += "---\n\n"

    return formatted

def format_conversation_history_example(history: List[Dict]) -> str:
    """Format conversation history (last 5 messages)"""
    if not history:
        return "No previous conversation."

    # Limit to last 5 messages
    recent = history[-5:]
    formatted = ""
    for msg in recent:
        role = msg.get("role", "").upper()
        content = msg.get("content", "")
        formatted += f"{role}: {content}\n"

    return formatted

# Example usage
sources = [
    Source(1, 1, "requirements.pdf", "requirements", "REQ-001: OAuth 2.0 support", 12, 0.9),
    Source(2, 1, "requirements.pdf", "requirements", "REQ-002: Token expiration", 13, 0.85)
]

history = [
    {"role": "user", "content": "What auth methods?"},
    {"role": "assistant", "content": "OAuth 2.0 and SAML 2.0"},
    {"role": "user", "content": "Tell me more about OAuth"}
]

print("FORMATTED CONTEXT:")
print(format_context_example(sources))

print("\nFORMATTED HISTORY:")
print(format_conversation_history_example(history))

FORMATTED CONTEXT:


AttributeError: 'Source' object has no attribute 'metadata'

### _call_llm()

```python
def _call_llm(self, prompt: str, context: AgentContext) -> str:
    """
    Call LLM with prompt
    """
```

#### Purpose

Handles the actual LLM API call with error handling.

#### Implementation

```python
try:
    response = self.llm_client.generate(
        prompt=prompt,
        temperature=self.temperature
    )
    return response
except OllamaException as e:
    logger.error(f"{self.name}: LLM call failed: {e}")
    raise
```

#### Error Handling

- Catches `OllamaException` (timeouts, connection errors, HTTP errors)
- Logs error with agent name for debugging
- Re-raises exception to be caught by `process_query()`

### _parse_response()

```python
def _parse_response(
    self,
    llm_response: str,
    retrieved_context: List[Source]
) -> AgentResponse:
    """Parse LLM response into structured format"""
```

#### Purpose

Converts the LLM's JSON response into a structured `AgentResponse` object.

#### Parsing Logic

**Step 1**: Clean response (remove markdown code blocks)
**Step 2**: Parse JSON
**Step 3**: Map confidence string to enum
**Step 4**: Build AgentResponse

#### Error Handling

If JSON parsing fails, returns a fallback response with:
- Raw LLM output as content
- LOW confidence
- Parse error in metadata

This ensures the user still sees the LLM's response even if not properly formatted.

In [15]:
def parse_response_example(llm_response: str, sources: List[Source]) -> dict:
    """Example of response parsing with error handling"""
    try:
        # Step 1: Clean response
        cleaned = llm_response.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned[7:]
        if cleaned.startswith("```"):
            cleaned = cleaned[3:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()

        # Step 2: Parse JSON
        parsed = json.loads(cleaned)

        # Step 3: Map confidence
        confidence_map = {
            "high": "HIGH",
            "medium": "MEDIUM",
            "low": "LOW"
        }
        confidence = confidence_map.get(
            parsed.get("confidence", "medium").lower(),
            "MEDIUM"
        )

        # Step 4: Build response
        return {
            "content": parsed.get("answer", ""),
            "confidence": confidence,
            "sources": len(sources),
            "metadata": {
                "key_sources": parsed.get("key_sources", []),
                "reasoning": parsed.get("reasoning", "")
            },
            "suggested_agents": parsed.get("suggests_consulting", []),
            "parse_success": True
        }

    except json.JSONDecodeError as e:
        print(f"⚠️  Failed to parse JSON: {e}")
        print(f"Response was: {llm_response[:100]}...")

        # Fallback response
        return {
            "content": llm_response,
            "confidence": "LOW",
            "sources": len(sources),
            "metadata": {"parse_error": str(e)},
            "suggested_agents": [],
            "parse_success": False
        }

# Test with valid JSON
valid_response = '''```json
{
  "answer": "OAuth 2.0 is supported per REQ-001 [Source 1]",
  "confidence": "high",
  "key_sources": ["requirements.pdf"],
  "suggests_consulting": [],
  "reasoning": "Clear requirement found"
}
```'''

result = parse_response_example(valid_response, sources)
print("Valid JSON parsing:")
print(json.dumps(result, indent=2))

# Test with invalid JSON
invalid_response = "This is not JSON at all"
result = parse_response_example(invalid_response, sources)
print("\nInvalid JSON parsing:")
print(json.dumps(result, indent=2))

Valid JSON parsing:
{
  "content": "OAuth 2.0 is supported per REQ-001 [Source 1]",
  "confidence": "HIGH",
  "sources": 2,
  "metadata": {
    "key_sources": [
      "requirements.pdf"
    ],
    "reasoning": "Clear requirement found"
  },
  "suggested_agents": [],
  "parse_success": true
}
⚠️  Failed to parse JSON: Expecting value: line 1 column 1 (char 0)
Response was: This is not JSON at all...

Invalid JSON parsing:
{
  "content": "This is not JSON at all",
  "confidence": "LOW",
  "sources": 2,
  "metadata": {
    "parse_error": "Expecting value: line 1 column 1 (char 0)"
  },
  "suggested_agents": [],
  "parse_success": false
}


### _calculate_keyword_score()

```python
def _calculate_keyword_score(self, query: str, keywords: List[str]) -> float:
    """
    Helper method to calculate relevance score based on keywords
    """
```

#### Purpose

Provides a simple keyword-based relevance scoring mechanism for implementing `can_handle_query()`.

#### Algorithm

1. Convert query to lowercase
2. Count keyword matches
3. Calculate score: `min(0.3 + (matches * 0.15), 1.0)`

#### Scoring Table

| Matches | Score |
|---------|-------|
| 0 | 0.0 |
| 1 | 0.45 |
| 2 | 0.60 |
| 3 | 0.75 |
| 4 | 0.90 |
| 5+ | 1.0 |

In [ ]:
def calculate_keyword_score(query: str, keywords: List[str]) -> float:
    """Calculate relevance score based on keyword matches"""
    query_lower = query.lower()
    matches = sum(1 for keyword in keywords if keyword.lower() in query_lower)

    if matches == 0:
        return 0.0

    # Base score of 0.3 + 0.15 per match, capped at 1.0
    score = min(0.3 + (matches * 0.15), 1.0)
    return score

# Example usage
keywords = ["requirement", "functional", "non-functional", "shall", "must", "req-"]

test_queries = [
    "What are the functional requirements?",
    "Show me requirement REQ-001",
    "Explain the authentication requirements and specifications",
    "What is the design pattern?",
    "List all requirements and functional specs"
]

for query in test_queries:
    score = calculate_keyword_score(query, keywords)
    print(f"Query: '{query}'")
    print(f"  Score: {score:.2f}\n")

## Subclass Implementation Guide

### Minimal Implementation

To create a specialized agent, implement the two abstract methods:

In [ ]:
class RequirementsAgent:
    """Example specialized agent implementation"""

    def __init__(self):
        self.agent_id = "requirements_agent"
        self.name = "Requirements Analysis Agent"
        self.category_id = 1
        self.temperature = 0.1
        print(f"✓ Initialized {self.name}")

    def get_system_prompt(self) -> str:
        return """You are a Requirements Analysis Expert.

Analyze software requirements documents focusing on:
- Functional requirements (REQ-FUNC-*)
- Non-functional requirements (REQ-NFR-*)
- Requirement clarity and testability
- Traceability information

Always cite specific requirement IDs."""

    def can_handle_query(self, query: str) -> float:
        keywords = [
            "requirement", "functional", "non-functional",
            "shall", "must", "req-", "specification"
        ]
        return calculate_keyword_score(query, keywords)

# Create instance
agent = RequirementsAgent()
print(f"Agent ID: {agent.agent_id}")
print(f"Category: {agent.category_id}")

# Test query handling
test_query = "What are the authentication requirements?"
score = agent.can_handle_query(test_query)
print(f"\nQuery: '{test_query}'")
print(f"Can handle score: {score:.2f}")

### Advanced Customization

Override additional methods for specialized behavior:

In [ ]:
class DesignAgent:
    """Example with advanced customization"""

    def __init__(self):
        self.agent_id = "design_agent"
        self.name = "Design Documentation Agent"
        self.category_id = 2
        self.temperature = 0.2  # Slightly higher for design explanations
        print(f"✓ Initialized {self.name}")

    def get_system_prompt(self) -> str:
        return """You are a Design Documentation Expert.

Specialize in:
- System architecture and design patterns
- Component interactions and interfaces
- UML diagrams and design models
- Design decisions and trade-offs

Reference architectural layers and design patterns."""

    def can_handle_query(self, query: str) -> float:
        keywords = ["design", "architecture", "pattern", "component", "interface", "uml"]
        return calculate_keyword_score(query, keywords)

    def retrieve_context_custom(self, query: str, top_k: int = 5):
        """Custom retrieval with design-specific boosting"""
        # Get base sources
        sources = retrieve_context_mock(query, self.category_id, self.name)

        # Boost architectural diagrams and UML sources
        for source in sources:
            if any(ext in source.filename for ext in ['.uml', '.drawio', 'architecture']):
                source.similarity_score *= 1.2
                print(f"  📈 Boosted {source.filename} to {source.similarity_score:.2f}")

        # Re-sort by boosted scores
        sources.sort(key=lambda s: s.similarity_score, reverse=True)
        return sources[:top_k]

# Test the design agent
design_agent = DesignAgent()
print(f"Temperature: {design_agent.temperature}")

query = "What design patterns are used?"
score = design_agent.can_handle_query(query)
print(f"\nQuery: '{query}'")
print(f"Can handle score: {score:.2f}")

## Error Handling Strategy

The BaseAgent implements comprehensive error handling at multiple levels:

### Level 1: LLM Call Failures

`_call_llm()` catches `OllamaException` and re-raises after logging.

### Level 2: Response Parsing Failures

`_parse_response()` catches `json.JSONDecodeError` and returns fallback response with raw LLM output.

### Level 3: Top-Level Exception Handling

`process_query()` catches all exceptions and returns error response with LOW confidence.

### Design Benefits

- **Never crashes**: Always returns a valid `AgentResponse`
- **Transparent errors**: Error messages preserved in response content and metadata
- **Graceful degradation**: Low confidence signals to orchestrator that response is unreliable
- **Debuggable**: Comprehensive logging at each failure point

In [ ]:
def demonstrate_error_handling():
    """Demonstrate error handling at different levels"""

    print("=== Error Handling Demonstration ===\n")

    # Level 1: LLM call failure (simulated)
    print("1. LLM Call Failure:")
    print("   - Catches: OllamaException")
    print("   - Action: Log error, re-raise")
    print("   - Result: Caught by process_query()\n")

    # Level 2: JSON parsing failure
    print("2. JSON Parsing Failure:")
    invalid_json = "This is not JSON"
    result = parse_response_example(invalid_json, [])
    print(f"   - Input: '{invalid_json}'")
    print(f"   - Confidence: {result['confidence']}")
    print(f"   - Parse success: {result['parse_success']}")
    print(f"   - User sees: Raw LLM output\n")

    # Level 3: Top-level exception
    print("3. Top-Level Exception Handling:")
    print("   - Catches: Any Exception")
    print("   - Returns: AgentResponse with:")
    print("     * content: 'Error: <error message>'")
    print("     * confidence: LOW")
    print("     * metadata: {'error': '<error details>'}")
    print("     * sources: []")
    print("   - Result: Orchestrator receives valid response\n")

    print("✓ No crashes, always returns valid response")
    print("✓ Errors are transparent and debuggable")

demonstrate_error_handling()

## Performance Considerations

### Processing Time Tracking

Every response includes `processing_time_ms` measured from the start of `process_query()` to completion.

This enables:
- Performance monitoring and alerting
- Identifying slow agents or queries
- Optimizing prompt size and retrieval depth
- SLA compliance tracking

### Conversation History Limiting

Only the last 5 messages are included in prompts to:
- Prevent token limit exhaustion
- Reduce LLM processing time
- Focus on recent, relevant context
- Control costs (tokens per request)

### Prompt Size Management

The `_format_context()` method could be enhanced to truncate long chunks for better performance.

## Testing Recommendations

### Unit Testing Abstract Methods

In [ ]:
class TestAgent:
    """Test agent for unit testing"""
    def __init__(self):
        self.agent_id = "test_agent"
        self.name = "Test Agent"
        self.category_id = 99

    def get_system_prompt(self) -> str:
        return "You are a test agent."

    def can_handle_query(self, query: str) -> float:
        return 0.5

def test_prompt_building():
    """Test prompt building functionality"""
    agent = TestAgent()
    system_prompt = agent.get_system_prompt()

    sources = [Source(1, 1, "test.pdf", "test", "Test content", 1, 0.9)]
    prompt = build_prompt_example("test query", sources, system_prompt)

    assert "You are a test agent" in prompt
    assert "test query" in prompt
    assert "[Source 1]" in prompt

    print("✓ Test passed: prompt_building")

test_prompt_building()

### Testing Error Handling

In [ ]:
def test_error_handling():
    """Test error handling scenarios"""

    # Test 1: JSON parsing with valid input
    valid = '{"answer": "Test", "confidence": "high"}'
    result = parse_response_example(valid, [])
    assert result["parse_success"] == True
    assert result["confidence"] == "HIGH"
    print("✓ Test passed: valid_json_parsing")

    # Test 2: JSON parsing with invalid input
    invalid = "Not JSON"
    result = parse_response_example(invalid, [])
    assert result["parse_success"] == False
    assert result["confidence"] == "LOW"
    assert "parse_error" in result["metadata"]
    print("✓ Test passed: invalid_json_parsing")

    # Test 3: Keyword scoring
    score = calculate_keyword_score("requirements spec", ["requirement", "spec"])
    assert score > 0.0
    print("✓ Test passed: keyword_scoring")

test_error_handling()

## Future Enhancement Opportunities

### Async/Await Support

Convert to async methods for concurrent agent execution:

```python
async def process_query(self, query: str, context: AgentContext) -> AgentResponse:
    retrieved_context = await self.retrieve_context_async(query, context)
    prompt = self._build_prompt(query, retrieved_context, context)
    response_text = await self._call_llm_async(prompt, context)
    return self._parse_response(response_text, retrieved_context)
```

### Response Caching

Cache responses for identical queries to improve performance.

### Prompt Template System

Use Jinja2 for flexible prompt templates.

### Metrics and Monitoring

Add instrumentation for production observability:
- Processing time histograms
- Success/error counters
- Confidence distribution tracking

### Structured Output with Pydantic

Replace JSON string parsing with structured outputs for better validation and type safety.

## Summary

The `BaseAgent` class provides a robust foundation for building specialized document analysis agents:

### Key Features

- ✅ **Abstract base class** enforcing consistent interface
- ✅ **Template method pattern** for flexible customization
- ✅ **Comprehensive error handling** with graceful degradation
- ✅ **Performance tracking** with processing time metrics
- ✅ **Extensible architecture** for specialized agents

### Implementation Requirements

Subclasses must implement:
1. `get_system_prompt()` - Define agent personality and expertise
2. `can_handle_query()` - Calculate relevance score for queries

### Optional Customizations

Override these for specialized behavior:
- `retrieve_context()` - Custom retrieval logic
- `_build_prompt()` - Enhanced prompt construction
- `_format_context()` - Specialized source formatting

### Integration

Agents are orchestrated to:
- Process queries with domain expertise
- Cite sources for transparency
- Suggest cross-agent collaboration
- Handle errors gracefully
- Track performance metrics